# Music 해상도(visual token) 실험 — 실습실 RTX 4090용

MMMU validation **Music 30문항**을 이미지 토큰 수만 바꿔서 다시 풉니다. 프롬프트, sampling(seed 42), 파서, judge, max_new_tokens(32768)는 **baseline(Run B)과 같습니다.**
평가 코드는 이 저장소의 `baseline/mmmu_eval/`을 **수정 없이** 불러오고, 해상도 값만 단계별로 바꿉니다.

| 단계 (Qwen3-VL 이미지당 토큰) | min_pixels = max_pixels | 28² 표기 | `max_model_len` (4090) |
|---|---|---|---|
| **441** | 451,584 px | 576×28² | 40960 |
| **784** | 802,816 px | 1024×28² | 40960 |
| **1,568** | 1,605,632 px | 2048×28² | 45056 |
| **3,920** | 4,014,080 px | 5120×28² | 57344 |
| (참고) baseline Run B | min 1,003,520 / max 4,014,080 px | 1280~5120×28² | 40960 |

- Qwen3-VL은 **32×32픽셀당 이미지 토큰 1개**입니다. **min = max**로 둬야 모든 이미지가 그 토큰 수로 맞춰집니다.
- `max_model_len` = 출력 한도 32,768 + 그 단계에서 가장 긴 프롬프트. 이미지 여러 장 문항(Music_1, 16, 23, 29)이 큰 단계에서 거부되지 않게 단계별로 잡았습니다. **출력 결과에는 영향이 없고**, 4090 메모리에 들어가도록 필요한 만큼만 늘렸습니다.
- 결과는 `results/music_resolution/music_tok_<토큰>/`에 저장됩니다. 중단 후 다시 실행하면 끝난 문항은 건너뜁니다.

실행 전 준비와 순서: `experiments/README_music_resolution_4090.md`

## 1. 설정

In [ ]:
import os, sys, json, time, signal, subprocess
from pathlib import Path
from argparse import Namespace
from dotenv import load_dotenv

# 저장소 루트 찾기 (이 노트북은 experiments/ 안에 있음)
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'baseline' / 'mmmu_eval').exists())

MODEL     = 'Qwen/Qwen3-VL-4B-Instruct'
REVISION  = 'ebb281ec70b05090aa6165b016eac8ec08e71b17'
JUDGE     = 'meta-llama/Llama-3.1-8B-Instruct'
JUDGE_REV = '0e9e39f249a16976918f6564b8830bc894c89659'

DATA_ROOT = REPO / 'data' / 'hf_mmmu'                 # MMMU Music 캐시 (Music config만 받음)
OUT_ROOT  = REPO / 'results' / 'music_resolution'     # 결과 저장 위치
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# 단계: 이미지당 토큰 -> 그 단계의 max_model_len (32768 + 가장 긴 프롬프트 + 여유)
LEVELS = {441: 40960, 784: 40960, 1568: 45056, 3920: 57344}
PIXELS = {t: t * 32 * 32 for t in LEVELS}
GPU_MEM_UTIL = 0.90                                   # 3,920 단계에서 KV cache 부족 오류가 나면 0.93~0.95로 올려 보기

load_dotenv(REPO / '.env')
if not os.environ.get('HF_TOKEN'):
    raise RuntimeError(f"HF_TOKEN이 없습니다. {REPO / '.env'} 파일에 HF_TOKEN=hf_... 한 줄을 넣으세요.")

sys.path.insert(0, str(REPO / 'baseline' / 'mmmu_eval'))
import run_mmmu_eval as R                             # baseline 평가 코드 그대로 사용

print('repo   :', REPO)
print('results:', OUT_ROOT)
print({t: f'{px:,} px' for t, px in PIXELS.items()})
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.used', '--format=csv'],
                     capture_output=True, text=True).stdout)

## 2. 서버 실행 도우미

In [ ]:
import requests

def kill_leftover_servers():
    """이전에 띄운 vLLM 서버가 남아 있으면 GPU 메모리가 모자라 새 서버가 뜨지 않는다."""
    subprocess.run(['pkill', '-f', 'vllm serve'], check=False)
    time.sleep(10)

def show_log_errors(log_path, tail=60):
    lines = Path(log_path).read_text(errors='ignore').splitlines()
    keys = ('error', 'out of memory', 'cuda', 'kv cache', 'max seq len', 'free memory')
    hits = [l for l in lines if any(k in l.lower() for k in keys)]
    print('--- 오류로 보이는 줄 ---')
    print('\n'.join(hits[-30:]) or '(없음)')
    print(f'--- 로그 마지막 {tail}줄 ---')
    print('\n'.join(lines[-tail:]))

def start_server(cmd, port, log_path):
    kill_leftover_servers()
    log = open(log_path, 'w')
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
                            env={**os.environ, 'VLLM_USE_V2_MODEL_RUNNER': '0'})
    for _ in range(720):                              # 최대 60분 (첫 실행은 모델 다운로드 포함)
        try:
            if requests.get(f'http://localhost:{port}/v1/models', timeout=5).ok:
                return proc
        except Exception:
            pass
        if proc.poll() is not None:
            show_log_errors(log_path)
            raise RuntimeError(f'vLLM 서버가 종료됐습니다. 전체 로그: {log_path}')
        time.sleep(5)
    stop_server(proc)
    raise RuntimeError(f'vLLM 서버가 제시간에 준비되지 않았습니다. 로그: {log_path}')

def stop_server(proc):
    try:
        os.killpg(proc.pid, signal.SIGTERM)
        proc.wait(timeout=120)
    except Exception:
        try:
            os.killpg(proc.pid, signal.SIGKILL)
        except Exception:
            pass
    time.sleep(10)                                    # GPU 메모리 해제 대기

## 3. 추론 — 단계마다 Qwen 서버를 새로 띄움
이전 단계 해상도로 처리된 이미지 캐시를 재사용하지 않도록 단계마다 서버를 다시 시작합니다.
특정 단계만 돌리려면 `RUN = [441, 3920]`처럼 바꾸세요.

In [ ]:
RUN = list(LEVELS)

for t in RUN:
    out = OUT_ROOT / f'music_tok_{t}'
    out.mkdir(parents=True, exist_ok=True)
    print(f'\n========== Music @ {t} tokens/image ({PIXELS[t]:,} px), max_model_len {LEVELS[t]} ==========')

    # baseline 코드의 해상도 상수만 이 단계 값으로 바꿈 (call_model이 mm_processor_kwargs로 전송)
    R.MIN_PIXELS = R.MAX_PIXELS = PIXELS[t]

    t0 = time.time()
    server = start_server(
        ['vllm', 'serve', MODEL, '--revision', REVISION, '--served-model-name', 'qwen3-vl-4b',
         '--dtype', 'bfloat16', '--seed', '42', '--max-model-len', str(LEVELS[t]),
         '--gpu-memory-utilization', str(GPU_MEM_UTIL), '--port', '8000'],
        8000, out / 'vllm_infer.log')
    try:
        R.cmd_infer(Namespace(
            output_dir=str(out), data_root=str(DATA_ROOT), subject='Music', all=False, limit=None,
            prompt='mmmu_pro_cot', max_new_tokens=32768, concurrency=32, request_timeout=3600,
            api_base='http://localhost:8000/v1', served_model_name='qwen3-vl-4b',
            model_path=MODEL, model_revision=REVISION))
    finally:
        stop_server(server)

    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip()
    (out / 'resolution.json').write_text(json.dumps({
        'tokens_per_image': t, 'min_pixels': PIXELS[t], 'max_pixels': PIXELS[t],
        'max_model_len': LEVELS[t], 'gpu_memory_utilization': GPU_MEM_UTIL, 'gpu': gpu,
        'wall_min_including_server_start': round((time.time() - t0) / 60, 1)}, indent=2))

## 4. judge + 채점 — judge 서버는 한 번만 띄워 모든 단계를 판정

In [ ]:
judge = start_server(
    ['vllm', 'serve', JUDGE, '--revision', JUDGE_REV, '--served-model-name', 'judge',
     '--dtype', 'bfloat16', '--seed', '42', '--max-model-len', '20480',
     '--gpu-memory-utilization', str(GPU_MEM_UTIL), '--port', '8001'],
    8001, OUT_ROOT / 'vllm_judge.log')
try:
    for t in RUN:
        R.cmd_judge(Namespace(output_dir=str(OUT_ROOT / f'music_tok_{t}'), subject='Music', all=False, limit=None,
                              judge_api_base='http://localhost:8001/v1', judge_model='judge',
                              judge_revision=JUDGE_REV, concurrency=16))
finally:
    stop_server(judge)

for t in RUN:
    R.MIN_PIXELS = R.MAX_PIXELS = PIXELS[t]           # summary.json에 이 단계 해상도가 기록되도록
    R.cmd_score(Namespace(output_dir=str(OUT_ROOT / f'music_tok_{t}'), subject='Music', all=False, limit=None))

## 5. 결과 비교
- baseline(Run B, A100)은 이 PC에 결과 파일이 없어도 비교할 수 있도록 Music 수치를 코드에 적어 두었습니다.
- **프롬프트 토큰 중앙값**이 단계별로 뚜렷하게 달라야 해상도가 실제로 적용된 것입니다.
- **오류 수가 0**이어야 합니다. 0이 아니면 `max_model_len` 부족 등으로 거부된 요청이 있다는 뜻입니다.
- Music 30문항은 샘플링 노이즈가 커서 ±몇 문항 차이는 우연일 수 있습니다. 잘림 수와 끝까지 답한 문항의 정답을 함께 보세요.

In [ ]:
# Run B Music: 5/30 정답, 잘림 15, 끝까지 답한 15문항 중 5문항 정답
BASELINE = {'correct': 5, 'n': 30, 'truncated': 15, 'done': '5/15', 'ok_ids': {3, 5, 12, 20, 27}}

def load(out_dir):
    rows = [json.loads(l) for l in open(out_dir / 'final_predictions.jsonl')]
    rows = [r for r in rows if r['subject'] == 'Music']
    done = [r for r in rows if r['finish_reason'] != 'length']
    pt = sorted(r['usage'].get('prompt_tokens', 0) for r in rows)
    return {
        'n': len(rows), 'correct': sum(r['correct'] for r in rows),
        'truncated': sum(r['finish_reason'] == 'length' for r in rows),
        'errors': sum(r['finish_reason'] == 'error' for r in rows),
        'done': f"{sum(r['correct'] for r in done)}/{len(done)}",
        'prompt_tok': pt[len(pt) // 2] if pt else 0,
        'ok_ids': {int(r['id'].split('_')[-1]) for r in rows if r['correct']},
    }

print(f"{'설정':<20}{'정답':>8}{'잘림':>6}{'오류':>6}{'끝까지 답한 문항 정답':>14}{'프롬프트 토큰 중앙값':>14}")
print(f"{'baseline Run B':<20}{BASELINE['correct']:>5}/{BASELINE['n']:<2}{BASELINE['truncated']:>6}{'-':>6}{BASELINE['done']:>14}{'약 1,100':>14}")
results = {}
for t in LEVELS:
    d = OUT_ROOT / f'music_tok_{t}'
    if (d / 'final_predictions.jsonl').exists():
        v = results[t] = load(d)
        print(f"{str(t) + ' tokens':<20}{v['correct']:>5}/{v['n']:<2}{v['truncated']:>6}{v['errors']:>6}{v['done']:>14}{v['prompt_tok']:>14}")

for t, v in results.items():
    print(f"\n{t} tokens: baseline 대비 새로 맞힘 {sorted(v['ok_ids'] - BASELINE['ok_ids'])} / 놓침 {sorted(BASELINE['ok_ids'] - v['ok_ids'])}")